In [1]:
import os
import sys
import json
import logging
import argparse
import torch
from scipy.sparse import issparse
import numpy as np
import anndata as ad
import pandas as pd

print("MPS available:", torch.backends.mps.is_available())
print("CUDA version:", torch.version.cuda)

# Show MPS device info safely
if torch.backends.mps.is_available():
    print("MPS backend is active on macOS (Metal).")
    print("Device: Apple GPU via Metal Performance Shaders (MPS).")
else:
    print("MPS not available — running on CPU instead.")

print("CPU cores:", os.cpu_count())


MPS available: False
CUDA version: 11.7
MPS not available — running on CPU instead.
CPU cores: 48


In [2]:
import scanpy as sc
# Load your dataset
adata_sc = sc.read_h5ad("./scrna-sciplex3/hvg.h5ad")

# Basic overview
print(adata_sc)
print("Shape:", adata_sc.shape)

# View column names (metadata about each cell)
print("Observation columns:", adata_sc.obs.columns.tolist()[:10])
print("Feature columns:", adata_sc.var_names[:10].tolist())

# How many drugs (conditions)?
print("Unique conditions:", adata_sc.obs['drug'].unique())
print("Number of cells per condition:")
print(adata_sc.obs['drug'].value_counts())



AnnData object with n_obs × n_vars = 762039 × 1000
    obs: 'size_factor', 'cell_type', 'replicate', 'dose', 'drug_code', 'pathway_level_1', 'pathway_level_2', 'product_name', 'target', 'pathway', 'drug', 'drug-dose', 'drug_code-dose', 'n_genes'
    var: 'gene_short_name', 'n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'hvg', 'pca', 'rank_genes_groups'
    obsm: 'X_pca'
    varm: 'PCs', 'marker_genes-drug-rank', 'marker_genes-drug-score'
Shape: (762039, 1000)
Observation columns: ['size_factor', 'cell_type', 'replicate', 'dose', 'drug_code', 'pathway_level_1', 'pathway_level_2', 'product_name', 'target', 'pathway']
Feature columns: ['ENSG00000243620.1', 'ENSG00000271503.5', 'ENSG00000259124.1', 'ENSG00000121101.15', 'ENSG00000160963.13', 'ENSG00000135346.8', 'ENSG00000143839.14', 'ENSG00000100867.14', 'ENSG00000140986.7', 'ENSG00000230666.5']
Unique conditions: ['tak_901', 'ag_490', 'abexinostat', 'alisertib', 'busulfan', ..., 'ac480', 'cediranib', 't

In [3]:
import os
import sys
import json
import logging
import argparse
import geomloss
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from tqdm import tqdm
import numpy as np
import pandas as pd
from pathlib import Path
from typing import Dict, Tuple, List, Optional
from umap import UMAP
import matplotlib.pyplot as plt
from scipy.optimize import linear_sum_assignment
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics.pairwise import rbf_kernel
from typing import Dict, Tuple, List
from scipy.stats import ks_2samp
from scipy.spatial.distance import cdist
from sklearn.metrics import r2_score

import gc
gc.collect()

def median_heuristic_gamma(X: np.ndarray, Y: np.ndarray) -> float:
    """
    Median heuristic for RBF bandwidth: gamma = 1 / median(||x - y||^2).
    Uses the median of pairwise distances in the pooled set.
    """
    Z = np.vstack([X, Y])
    # Sample if too large for efficiency
    max_samples = 5000
    if Z.shape[0] > max_samples:
        idx = np.random.choice(Z.shape[0], size=max_samples, replace=False)
        Z = Z[idx]
    D2 = cdist(Z, Z, metric='sqeuclidean')
    # Use upper triangular without diagonal
    triu = D2[np.triu_indices_from(D2, k=1)]
    med = np.median(triu[triu > 0]) if np.any(triu > 0) else 1.0
    return 1.0 / max(med, 1e-12)

def mmd_distance(X: np.ndarray, Y: np.ndarray, gamma: float) -> float:
    """
    Unbiased MMD^2 estimator using Gaussian (RBF) kernel, sklearn backend.

    Args:
        X: (n_samples, n_features) first sample
        Y: (m_samples, n_features) second sample
        gamma: RBF kernel bandwidth; if None, uses median heuristic

    Returns:
        Unbiased MMD^2 value
    """
    n = X.shape[0]
    m = Y.shape[0]

    # Kernel matrices
    Kxx = rbf_kernel(X, X, gamma=gamma)
    Kyy = rbf_kernel(Y, Y, gamma=gamma)
    Kxy = rbf_kernel(X, Y, gamma=gamma)

    # Unbiased: exclude diagonal entries
    np.fill_diagonal(Kxx, 0.0)
    np.fill_diagonal(Kyy, 0.0)

    term_xx = Kxx.sum() / (n * (n - 1)) if n > 1 else 0.0
    term_yy = Kyy.sum() / (m * (m - 1)) if m > 1 else 0.0
    term_xy = 2.0 * Kxy.mean()

    mmd2 = term_xx + term_yy - term_xy
    mmd2 = max(mmd2, 0.0)  # Numerical stability
    return float(mmd2)

def r2_feature_means(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """
    R^2 computed across features between mean vectors of y_true and y_pred.
    """
    mu_true = y_true.mean(axis=0)
    mu_pred = y_pred.mean(axis=0)
    ss_res = float(np.sum((mu_true - mu_pred) ** 2))
    ss_tot = float(np.sum((mu_true - mu_true.mean()) ** 2))
    if ss_tot <= 1e-12:
        return 1.0 if ss_res <= 1e-12 else 0.0
    return 1.0 - ss_res / ss_tot

def wasserstein_pointcloud(
    X,
    Y,
    p: int = 2,
    a=None,
    b=None,
    method: str = "emd",          # "emd" (exact) or "sinkhorn" (approx)
    reg: float = 1e-1,            # Sinkhorn regularization (only used if method="sinkhorn")
    return_plan: bool = False,
):
    """
    Compute Wasserstein distance W_p between two empirical distributions supported on point sets X and Y.

    Parameters
    ----------
    X : (n, d) array-like
        Source points.
    Y : (m, d) array-like
        Target points.
    p : int
        Order of the Wasserstein distance (commonly 1 or 2).
    a : (n,) array-like or None
        Weights for X; if None, uniform weights.
    b : (m,) array-like or None
        Weights for Y; if None, uniform weights.
    method : str
        "emd" for exact optimal transport (requires POT),
        "sinkhorn" for entropic approximation (requires POT).
    reg : float
        Entropic regularization strength for Sinkhorn.
    return_plan : bool
        If True, also return the optimal transport plan.

    Returns
    -------
    Wp : float
        Wasserstein distance of order p.
    plan : (n, m) ndarray, optional
        Optimal transport plan (only if return_plan=True).
    """
    X = np.asarray(X, dtype=np.float64)
    Y = np.asarray(Y, dtype=np.float64)
    if X.ndim != 2 or Y.ndim != 2:
        raise ValueError("X and Y must be 2D arrays with shape (n, d) and (m, d).")
    if X.shape[1] != Y.shape[1]:
        raise ValueError(f"Dimension mismatch: X has d={X.shape[1]}, Y has d={Y.shape[1]}.")

    n, d = X.shape
    m, _ = Y.shape

    if a is None:
        a = np.full(n, 1.0 / n, dtype=np.float64)
    else:
        a = np.asarray(a, dtype=np.float64)
        a = a / a.sum()

    if b is None:
        b = np.full(m, 1.0 / m, dtype=np.float64)
    else:
        b = np.asarray(b, dtype=np.float64)
        b = b / b.sum()

    # Cost matrix: C_ij = ||x_i - y_j||^p
    # Compute squared Euclidean via (x-y)^2 = x^2 + y^2 - 2xy for speed
    X2 = np.sum(X * X, axis=1, keepdims=True)          # (n, 1)
    Y2 = np.sum(Y * Y, axis=1, keepdims=True).T        # (1, m)
    sq = np.maximum(X2 + Y2 - 2.0 * (X @ Y.T), 0.0)     # (n, m)
    if p == 2:
        C = sq
    else:
        C = sq ** (p / 2.0)

    try:
        import ot  # POT: Python Optimal Transport
    except ImportError as e:
        raise ImportError(
            "This function requires the POT library. Install with: pip install pot"
        ) from e

    method = method.lower()
    if method == "emd":
        # exact OT: minimizes <P, C>
        P = ot.emd(a, b, C)
        cost = float(np.sum(P * C))
    elif method == "sinkhorn":
        # entropic OT approximation
        P = ot.sinkhorn(a, b, C, reg=reg)
        cost = float(np.sum(P * C))
    else:
        raise ValueError('method must be either "emd" or "sinkhorn".')

    Wp = cost ** (1.0 / p)

    if return_plan:
        return Wp, P
    return Wp

def summarize_metrics(y_true: np.ndarray, y_pred: np.ndarray, median_gamma: float) -> dict:
    """
    Compute a standard set of metrics: MMD^2 (RBF), R^2 of feature means, median KS across features, and Wasserstein distance.
    """
    # Drop any samples that contain NaNs in either true or pred
    mask = (~np.isnan(y_true).any(axis=1)) & (~np.isnan(y_pred).any(axis=1))
    if mask.sum() < len(y_true):
        print(f"[summarize_metrics] Dropping {len(y_true) - mask.sum()} samples with NaNs.")
    
    y_true = y_true[mask]
    y_pred = y_pred[mask]

    out = {}

    out['mmd2_gamma_median'] = mmd_distance(y_true, y_pred, gamma=median_gamma)
    out['mmd2_gamma_0.5'] = mmd_distance(y_true, y_pred, gamma=0.5)
    out['mmd2_gamma_1.0'] = mmd_distance(y_true, y_pred, gamma=1.0)
    out['wasserstein_distance'] = wasserstein_pointcloud(y_true, y_pred, p=2, method="emd")
    out['R2_feature_means'] = r2_feature_means(y_true, y_pred)
    return out

def split_train_test(X: np.ndarray, Y: np.ndarray, train_fraction: float, seed: int = 42) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    if X.shape[0] != Y.shape[0]:
        min_len = min(len(X), len(Y))
        X = X[:min_len]
        Y = Y[:min_len]

    n = X.shape[0]
    n_train = max(1, int(n * train_fraction))
    rng = np.random.default_rng(seed)
    idx = rng.permutation(n)
    tr_idx, te_idx = idx[:n_train], idx[n_train:]
    return X[tr_idx], X[te_idx], Y[tr_idx], Y[te_idx]

def topk_markers(adata, drug: str, k: int = 50, rank_key: str = "marker_genes-drug-rank"):
    R = adata.varm[rank_key]

    # --- get the rank vector for this drug ---
    if hasattr(R, "columns") and hasattr(R, "iloc"):  # pandas DataFrame
        if drug in R.columns:
            r = R[drug].to_numpy()
        else:
            # fallback: interpret columns as ordered groups; try to map via rank_genes_groups names
            names = adata.uns["rank_genes_groups"]["names"]
            groups = list(names.dtype.names) if (hasattr(names, "dtype") and names.dtype.names is not None) else list(names.columns)
            r = R.iloc[:, groups.index(drug)].to_numpy()
    else:  # numpy array (or array-like)
        names = adata.uns["rank_genes_groups"]["names"]
        groups = list(names.dtype.names) if (hasattr(names, "dtype") and names.dtype.names is not None) else list(names.columns)
        r = np.asarray(R)[:, groups.index(drug)]

    # smaller rank => stronger marker
    idx = np.argsort(r)[:k]
    gene_ids = adata.var_names[idx].to_list()
    gene_short = (adata.var.iloc[idx]["gene_short_name"].to_list()
                  if "gene_short_name" in adata.var.columns else None)
    return gene_ids, gene_short, idx


In [4]:
def SCGEN(
    X_tr_pre, Y_tr_post, X_te_pre, Y_te_post,
    max_epochs=1000,
    batch_size=64,
    early_stopping=True,
    early_stopping_patience=50,
    condition_key="condition",
    ctrl_label="control",
    stim_label="treated",
    cell_type_key="cell_type",
    cell_type_label="cell",
    n_hidden=256,
    n_latent=100,
    n_layers=2,
    dropout_rate=0.2,
    accelerator="auto",   # "auto" | "cpu" | "gpu" | "mps"
    learning_rate=5e-5,
    seed=12345,
    verbose=True,
    metrics_fn=None,      # e.g., summarize_metrics(y_true, y_pred)
    top_feature_subset= None,
):
    import numpy as np
    import pandas as pd
    import random

    try:
        import anndata as ad
        import torch
        import scgen
        try:
            import scvi
        except Exception:
            scvi = None
    except Exception as e:
        msg = str(e)
        hint = None
        if "scvi._compat" in msg:
            hint = (
                "Likely scGen (PyPI) vs scvi-tools mismatch. "
                "Try: pip uninstall -y scgen && pip install git+https://github.com/theislab/scgen.git"
            )
        return {"error": f"Import failed: {e}", "hint": hint}

    # ----------- Validate shapes -----------
    X_tr_pre = np.asarray(X_tr_pre); Y_tr_post = np.asarray(Y_tr_post)
    X_te_pre = np.asarray(X_te_pre); Y_te_post = np.asarray(Y_te_post)

    if any(a.ndim != 2 for a in [X_tr_pre, Y_tr_post, X_te_pre, Y_te_post]):
        return {"error": "All inputs must be 2D arrays (n_cells, n_features)."}
    d = X_tr_pre.shape[1]
    if any(a.shape[1] != d for a in [Y_tr_post, X_te_pre, Y_te_post]):
        return {"error": "Feature dimension mismatch across inputs."}

    # ----------- Seeds -----------
    random.seed(seed); np.random.seed(seed)
    try:
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
    except Exception:
        pass
    if scvi is not None:
        try:
            scvi.settings.seed = seed
        except Exception:
            pass

    # ----------- Accelerator selection -----------
    def _auto_accel():
        if torch.cuda.is_available():
            return "cuda"
        if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
            return "mps"
        return "cpu"
    accelerator_used = _auto_accel() if accelerator == "auto" else accelerator

    # ----------- Build training AnnData -----------
    try:
        X_tr_all = np.vstack([X_tr_pre, Y_tr_post]).astype(np.float32, copy=False)
        cond_tr = np.array([ctrl_label] * len(X_tr_pre) + [stim_label] * len(Y_tr_post), dtype=object)
        ctype_tr = np.array([cell_type_label] * len(X_tr_all), dtype=object)

        ad_tr = ad.AnnData(
            X_tr_all,
            obs=pd.DataFrame({condition_key: cond_tr, cell_type_key: ctype_tr})
        )
        # avoid "Observation names are not unique"
        ad_tr.obs_names = [f"tr_{i}" for i in range(ad_tr.n_obs)]
        ad_tr.var_names = [f"g_{j}" for j in range(ad_tr.n_vars)]

        scgen.SCGEN.setup_anndata(ad_tr, batch_key=condition_key, labels_key=cell_type_key)

        model = scgen.SCGEN(
            ad_tr,
            n_hidden=n_hidden,
            n_latent=n_latent,
            n_layers=n_layers,
            dropout_rate=dropout_rate,
        )

        train_kwargs = dict(
            max_epochs=int(max_epochs),
            batch_size=int(batch_size),
            early_stopping=bool(early_stopping),
            early_stopping_patience=int(early_stopping_patience),
            enable_progress_bar=bool(verbose),
            plan_kwargs={"lr": float(learning_rate)},
        )


        model.train(**train_kwargs)

    except Exception as e:
        return {"error": f"scGen training failed: {e}"}

    # ----------- Predict treated for test controls -----------

    ad_te = ad.AnnData(
        X_te_pre.astype(np.float32, copy=False),
        obs=pd.DataFrame({
            condition_key: np.array([ctrl_label] * len(X_te_pre), dtype=object),
            cell_type_key: np.array([cell_type_label] * len(X_te_pre), dtype=object),
        })
    )
    ad_te.obs_names = [f"te_{i}" for i in range(ad_te.n_obs)]
    ad_te.var_names = ad_tr.var_names.copy()  # ensure exact match

    # IMPORTANT: pass ONLY adata_to_predict OR celltype_to_predict (not both). :contentReference[oaicite:3]{index=3}
    pred_adata, delta = model.predict(
        ctrl_key=ctrl_label,
        stim_key=stim_label,
        adata_to_predict=ad_te,
    )

    y_pred = np.asarray(pred_adata.X)

    return {
        "y_pred": y_pred,
        "delta": delta,
        "model": model,
        "adata_train": ad_tr,
        "accelerator_used": accelerator_used,
    }




In [5]:
drug = "trametinib"
X_pre = adata_sc[adata_sc.obs["drug"] == "control"].copy().to_df()
X_post  = adata_sc[adata_sc.obs["drug"] == drug].copy().to_df()

print("X_pre cells:", X_pre.shape)
print("X_post cells:", X_post.shape)

top_genes_ids, top_genes_short, top_genes_idx = topk_markers(adata_sc, drug, k=100)

X_tr_pre, X_te_pre, Y_tr_post, Y_te_post = split_train_test(X_pre.values, X_post.values, 0.8)

print(X_tr_pre.shape)
print(X_te_pre.shape)
print(Y_tr_post.shape)
print(Y_te_post.shape)


# Compute median heuristic gamma on training data
median_gamma = median_heuristic_gamma(X_tr_pre, Y_tr_post)
print("Median heuristic gamma:", median_gamma)


X_pre cells: (17565, 1000)
X_post cells: (3277, 1000)
(2621, 1000)
(656, 1000)
(2621, 1000)
(656, 1000)
Median heuristic gamma: 0.0022819381995053574


In [6]:
# Multiple runs for robustness
all_metrics = []
for run in range(10):
    print(f"**************** Run: {run} ****************")
    seed = 1234 + run
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    out = SCGEN(X_tr_pre[:, top_genes_idx], Y_tr_post[:, top_genes_idx], X_te_pre[:, top_genes_idx], Y_te_post[:, top_genes_idx], max_epochs=1000,early_stopping_patience=50,learning_rate=1e-3, batch_size= 256, early_stopping=20, dropout_rate=0, seed=seed)
    metrics = summarize_metrics(out["y_pred"][:, :], Y_te_post[:, top_genes_idx], median_gamma)
    print("Metrics:", metrics)
    all_metrics.append(metrics)

# Results summary
df = pd.DataFrame(all_metrics)
print("=== Metrics Summary over Runs for top 100 genes ===")
print(df.describe().T[['mean', 'std']].round(4))

**************** Run: 0 ****************


/u/jrp5td/here/miniconda3/envs/scgen-env/lib/python3.9/site-packages/pytorch_lightning/utilities/imports.py:22: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Global seed set to 0
/u/jrp5td/here/miniconda3/envs/scgen-env/lib/python3.9/site-packages/pytorch_lightning/utilities/warnings.py:53: LightningDeprecationWarning: pytorch_lightning.utilities.warnings.rank_zero_deprecation has been deprecated in v1.6 and will be removed in v1.8. Use the equivalent function from the pytorch_lightning.utilities.rank_zero module instead.
  new_rank_zero_deprecation(
/u/jrp5td/here/miniconda3/envs/scgen-env/lib/python3.9/site-packages/pytorch_lightning/utilities/warnings.py:58: LightningDeprecationWarning: The `pytorch_lightning.loggers.base.rank_zero_experiment` is deprecated in v1.7 an

Epoch 51/1000:   5%|██████████▋                                                                                                                                                                                                       | 51/1000 [00:05<01:42,  9.28it/s, loss=2.03, v_num=1]
Monitored metric elbo_validation did not improve in the last 50 records. Best score: 58.538. Signaling Trainer to stop.
INFO     Received view of anndata, making copy.                                                                    
INFO     Input AnnData not setup with scvi-tools. attempting to transfer AnnData setup                             
INFO     Received view of anndata, making copy.                                                                    
INFO     Input AnnData not setup with scvi-tools. attempting to transfer AnnData setup                             
INFO     Input AnnData not setup with scvi-tools. attempting to transfer AnnData setup                             


/u/jrp5td/here/miniconda3/envs/scgen-env/lib/python3.9/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/u/jrp5td/here/miniconda3/envs/scgen-env/lib/python3.9/site-packages/anndata/_core/anndata.py:1828: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
Global seed set to 1235


Metrics: {'mmd2_gamma_median': 0.007178729269851303, 'mmd2_gamma_0.5': 0.0001868965798021635, 'mmd2_gamma_1.0': 0.00011129798702495092, 'wasserstein_distance': 6.985626268083184, 'R2_feature_means': 0.5457381494170639}
**************** Run: 1 ****************


...

/u/jrp5td/here/miniconda3/envs/scgen-env/lib/python3.9/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Epoch 51/1000:   5%|██████████▋                                                                                                                                                                                                       | 51/1000 [00:11<03:38,  4.34it/s, loss=1.95, v_num=1]
Monitored metric elbo_validation did not improve in the last 50 records. Best score: 57.150. Signaling Trainer to stop.
INFO     Received view of anndata, making copy.                                                                    
INFO     Input AnnData not setup with scvi-tools. attempting to transfer AnnData setup                             
INFO     Received view of anndata, making copy.                                                                    
INFO     Input AnnData not setup with scvi-tools. attempting to transfer AnnData setup                             
INFO     Input AnnData not setup with scvi-tools. attempting to transfer AnnData setup                             
Metrics: {'mmd2

/u/jrp5td/here/miniconda3/envs/scgen-env/lib/python3.9/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/u/jrp5td/here/miniconda3/envs/scgen-env/lib/python3.9/site-packages/anndata/_core/anndata.py:1828: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
Global seed set to 1243
/u/jrp5td/here/miniconda3/envs/scgen-env/lib/python3.9/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Epoch 51/1000:   5%|██████████▋                                                                                                                                                                                                       | 51/1000 [00:05<01:42,  9.22it/s, loss=1.95, v_num=1]
Monitored metric elbo_validation did not improve in the last 50 records. Best score: 60.970. Signaling Trainer to stop.
INFO     Received view of anndata, making copy.                                                                    
INFO     Input AnnData not setup with scvi-tools. attempting to transfer AnnData setup                             
INFO     Received view of anndata, making copy.                                                                    
INFO     Input AnnData not setup with scvi-tools. attempting to transfer AnnData setup                             
INFO     Input AnnData not setup with scvi-tools. attempting to transfer AnnData setup                             
Metrics: {'mmd2

/u/jrp5td/here/miniconda3/envs/scgen-env/lib/python3.9/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/u/jrp5td/here/miniconda3/envs/scgen-env/lib/python3.9/site-packages/anndata/_core/anndata.py:1828: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [7]:
drug = "givinostat"
X_pre = adata_sc[adata_sc.obs["drug"] == "control"].copy().to_df()
X_post  = adata_sc[adata_sc.obs["drug"] == drug].copy().to_df()

print("X_pre cells:", X_pre.shape)
print("X_post cells:", X_post.shape)

top_genes_ids, top_genes_short, top_genes_idx = topk_markers(adata_sc, drug, k=100)

X_tr_pre, X_te_pre, Y_tr_post, Y_te_post = split_train_test(X_pre.values, X_post.values, 0.8)

print(X_tr_pre.shape)
print(X_te_pre.shape)
print(Y_tr_post.shape)
print(Y_te_post.shape)


# Compute median heuristic gamma on training data
median_gamma = median_heuristic_gamma(X_tr_pre, Y_tr_post)
print("Median heuristic gamma:", median_gamma)


X_pre cells: (17565, 1000)
X_post cells: (3541, 1000)
(2832, 1000)
(709, 1000)
(2832, 1000)
(709, 1000)
Median heuristic gamma: 0.0022144700418584803


In [9]:
# Multiple runs for robustness
all_metrics = []
for run in range(10):
    print(f"**************** Run: {run} ****************")
    seed = 1234 + run
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    out = SCGEN(X_tr_pre[:, top_genes_idx], Y_tr_post[:, top_genes_idx], X_te_pre[:, top_genes_idx], Y_te_post[:, top_genes_idx], max_epochs=1000,early_stopping_patience=50,learning_rate=1e-3, batch_size= 256, early_stopping=20, dropout_rate=0, seed=seed)
    metrics = summarize_metrics(out["y_pred"][:, :], Y_te_post[:, top_genes_idx], median_gamma)
    print("Metrics:", metrics)
    all_metrics.append(metrics)

# Results summary
df = pd.DataFrame(all_metrics)
print("=== Metrics Summary over Runs for top 100 genes ===")
print(df.describe().T[['mean', 'std']].round(4))

Global seed set to 1234


**************** Run: 0 ****************


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Epoch 51/1000:   5%|██████████▋                                                                                                                                                                                                       | 51/1000 [00:12<03:48,  4.15it/s, loss=2.13, v_num=1]
Monitored metric elbo_validation did not improve in the last 50 records. Best score: 66.223. Signaling Trainer to stop.
INFO     Received view of anndata, making copy.                                                                    
INFO     Input AnnData not setup with scvi-tools. attempting to transfer AnnData setup                             
INFO     Received view of anndata, making copy.                                                                    
INFO     Input AnnData not setup with scvi-tools. attempting to transfer AnnData setup                             
INFO     Input AnnData not setup with scvi-tools. attempting to transfer AnnData setup                             


/u/jrp5td/here/miniconda3/envs/scgen-env/lib/python3.9/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/u/jrp5td/here/miniconda3/envs/scgen-env/lib/python3.9/site-packages/anndata/_core/anndata.py:1828: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
Global seed set to 1235


...

/u/jrp5td/here/miniconda3/envs/scgen-env/lib/python3.9/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/u/jrp5td/here/miniconda3/envs/scgen-env/lib/python3.9/site-packages/anndata/_core/anndata.py:1828: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
Global seed set to 1243


Metrics: {'mmd2_gamma_median': 0.006372802771999808, 'mmd2_gamma_0.5': 0.00034521305617024774, 'mmd2_gamma_1.0': 0.00016882490760494507, 'wasserstein_distance': 7.49713472030701, 'R2_feature_means': 0.8277694693738168}
**************** Run: 9 ****************


/u/jrp5td/here/miniconda3/envs/scgen-env/lib/python3.9/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Epoch 51/1000:   5%|██████████▊                                                                                                                                                                                                        | 51/1000 [00:05<01:46,  8.87it/s, loss=2.1, v_num=1]
Monitored metric elbo_validation did not improve in the last 50 records. Best score: 63.642. Signaling Trainer to stop.
INFO     Received view of anndata, making copy.                                                                    
INFO     Input AnnData not setup with scvi-tools. attempting to transfer AnnData setup                             
INFO     Received view of anndata, making copy.                                                                    
INFO     Input AnnData not setup with scvi-tools. attempting to transfer AnnData setup                             
INFO     Input AnnData not setup with scvi-tools. attempting to transfer AnnData setup                             
Metrics: {'mmd2

/u/jrp5td/here/miniconda3/envs/scgen-env/lib/python3.9/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/u/jrp5td/here/miniconda3/envs/scgen-env/lib/python3.9/site-packages/anndata/_core/anndata.py:1828: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [10]:
drug = "abexinostat"
X_pre = adata_sc[adata_sc.obs["drug"] == "control"].copy().to_df()
X_post  = adata_sc[adata_sc.obs["drug"] == drug].copy().to_df()

print("X_pre cells:", X_pre.shape)
print("X_post cells:", X_post.shape)

top_genes_ids, top_genes_short, top_genes_idx = topk_markers(adata_sc, drug, k=100)

X_tr_pre, X_te_pre, Y_tr_post, Y_te_post = split_train_test(X_pre.values, X_post.values, 0.8)

print(X_tr_pre.shape)
print(X_te_pre.shape)
print(Y_tr_post.shape)
print(Y_te_post.shape)


# Compute median heuristic gamma on training data
median_gamma = median_heuristic_gamma(X_tr_pre, Y_tr_post)
print("Median heuristic gamma:", median_gamma)


X_pre cells: (17565, 1000)
X_post cells: (4505, 1000)
(3604, 1000)
(901, 1000)
(3604, 1000)
(901, 1000)
Median heuristic gamma: 0.0021463110496127745


In [11]:
# Multiple runs for robustness
all_metrics = []
for run in range(10):
    print(f"**************** Run: {run} ****************")
    seed = 1234 + run
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    out = SCGEN(X_tr_pre[:, top_genes_idx], Y_tr_post[:, top_genes_idx], X_te_pre[:, top_genes_idx], Y_te_post[:, top_genes_idx], max_epochs=1000,early_stopping_patience=50,learning_rate=1e-3, batch_size= 256, early_stopping=20, dropout_rate=0, seed=seed)
    metrics = summarize_metrics(out["y_pred"][:, :], Y_te_post[:, top_genes_idx], median_gamma)
    print("Metrics:", metrics)
    all_metrics.append(metrics)

# Results summary
df = pd.DataFrame(all_metrics)
print("=== Metrics Summary over Runs for top 100 genes ===")
print(df.describe().T[['mean', 'std']].round(4))

Global seed set to 1234


**************** Run: 0 ****************


/u/jrp5td/here/miniconda3/envs/scgen-env/lib/python3.9/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Epoch 51/1000:   5%|██████████▋                                                                                                                                                                                                       | 51/1000 [00:06<02:04,  7.60it/s, loss=1.83, v_num=1]
Monitored metric elbo_validation did not improve in the last 50 records. Best score: 61.081. Signaling Trainer to stop.
INFO     Received view of anndata, making copy.                                                                    
INFO     Input AnnData not setup with scvi-tools. attempting to transfer AnnData setup                             
INFO     Received view of anndata, making copy.                                                                    
INFO     Input AnnData not setup with scvi-tools. attempting to transfer AnnData setup                             
INFO     Input AnnData not setup with scvi-tools. attempting to transfer AnnData setup                             


/u/jrp5td/here/miniconda3/envs/scgen-env/lib/python3.9/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/u/jrp5td/here/miniconda3/envs/scgen-env/lib/python3.9/site-packages/anndata/_core/anndata.py:1828: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
Global seed set to 1235


...

Metrics: {'mmd2_gamma_median': 0.0043786293987810865, 'mmd2_gamma_0.5': 3.499790119894403e-05, 'mmd2_gamma_1.0': 2.0828849805826214e-05, 'wasserstein_distance': 7.5412773076501605, 'R2_feature_means': 0.8363971273065126}
**************** Run: 9 ****************


/u/jrp5td/here/miniconda3/envs/scgen-env/lib/python3.9/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Epoch 51/1000:   5%|██████████▋                                                                                                                                                                                                       | 51/1000 [00:06<02:07,  7.43it/s, loss=1.86, v_num=1]
Monitored metric elbo_validation did not improve in the last 50 records. Best score: 59.337. Signaling Trainer to stop.
INFO     Received view of anndata, making copy.                                                                    
INFO     Input AnnData not setup with scvi-tools. attempting to transfer AnnData setup                             
INFO     Received view of anndata, making copy.                                                                    
INFO     Input AnnData not setup with scvi-tools. attempting to transfer AnnData setup                             
INFO     Input AnnData not setup with scvi-tools. attempting to transfer AnnData setup                             


/u/jrp5td/here/miniconda3/envs/scgen-env/lib/python3.9/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/u/jrp5td/here/miniconda3/envs/scgen-env/lib/python3.9/site-packages/anndata/_core/anndata.py:1828: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


Metrics: {'mmd2_gamma_median': 0.0035178527539190263, 'mmd2_gamma_0.5': 3.809052969061726e-05, 'mmd2_gamma_1.0': 2.2511289173081983e-05, 'wasserstein_distance': 7.514659877139623, 'R2_feature_means': 0.8712491723685304}
=== Metrics Summary over Runs for top 100 genes ===
                        mean     std
mmd2_gamma_median     0.0042  0.0006
mmd2_gamma_0.5        0.0000  0.0000
mmd2_gamma_1.0        0.0000  0.0000
wasserstein_distance  7.5380  0.0602
R2_feature_means      0.8351  0.0358


In [13]:
# Multiple runs for robustness
all_metrics = []
for run in range(10):
    print(f"**************** Run: {run} ****************")
    seed = 1234 + run
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    out = SCGEN(X_tr_pre[:, ], Y_tr_post[:, ], X_te_pre[:, ], Y_te_post[:, ], max_epochs=1000,early_stopping_patience=50,learning_rate=1e-3, batch_size= 256, early_stopping=20, dropout_rate=0, seed=seed)
    metrics = summarize_metrics(out["y_pred"][:, top_genes_idx], Y_te_post[:, top_genes_idx], median_gamma)
    print("Metrics:", metrics)
    all_metrics.append(metrics)

# Results summary
df = pd.DataFrame(all_metrics)
print("=== Metrics Summary over Runs for top 100 genes ===")
print(df.describe().T[['mean', 'std']].round(4))

Global seed set to 1234


**************** Run: 0 ****************


/u/jrp5td/here/miniconda3/envs/scgen-env/lib/python3.9/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Epoch 51/1000:   5%|██████████▋                                                                                                                                                                                                       | 51/1000 [00:07<02:19,  6.81it/s, loss=56.5, v_num=1]
Monitored metric elbo_validation did not improve in the last 50 records. Best score: 288.650. Signaling Trainer to stop.
INFO     Received view of anndata, making copy.                                                                    
INFO     Input AnnData not setup with scvi-tools. attempting to transfer AnnData setup                             
INFO     Received view of anndata, making copy.                                                                    
INFO     Input AnnData not setup with scvi-tools. attempting to transfer AnnData setup                             
INFO     Input AnnData not setup with scvi-tools. attempting to transfer AnnData setup                             


/u/jrp5td/here/miniconda3/envs/scgen-env/lib/python3.9/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/u/jrp5td/here/miniconda3/envs/scgen-env/lib/python3.9/site-packages/anndata/_core/anndata.py:1828: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
Global seed set to 1235


...

Metrics: {'mmd2_gamma_median': 0.011166420744414873, 'mmd2_gamma_0.5': 0.00027186881533112876, 'mmd2_gamma_1.0': 1.2589257222048309e-05, 'wasserstein_distance': 6.903050438915976, 'R2_feature_means': 0.5857998375991802}
**************** Run: 9 ****************


/u/jrp5td/here/miniconda3/envs/scgen-env/lib/python3.9/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Epoch 51/1000:   5%|██████████▊                                                                                                                                                                                                         | 51/1000 [00:06<02:08,  7.39it/s, loss=56, v_num=1]
Monitored metric elbo_validation did not improve in the last 50 records. Best score: 278.714. Signaling Trainer to stop.
INFO     Received view of anndata, making copy.                                                                    
INFO     Input AnnData not setup with scvi-tools. attempting to transfer AnnData setup                             
INFO     Received view of anndata, making copy.                                                                    
INFO     Input AnnData not setup with scvi-tools. attempting to transfer AnnData setup                             
INFO     Input AnnData not setup with scvi-tools. attempting to transfer AnnData setup                             


/u/jrp5td/here/miniconda3/envs/scgen-env/lib/python3.9/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/u/jrp5td/here/miniconda3/envs/scgen-env/lib/python3.9/site-packages/anndata/_core/anndata.py:1828: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


Metrics: {'mmd2_gamma_median': 0.009560534549209354, 'mmd2_gamma_0.5': 0.00023838204644722148, 'mmd2_gamma_1.0': 9.782957914069552e-06, 'wasserstein_distance': 6.779318405322399, 'R2_feature_means': 0.7534238301195895}
=== Metrics Summary over Runs for top 100 genes ===
                        mean     std
mmd2_gamma_median     0.0091  0.0012
mmd2_gamma_0.5        0.0002  0.0000
mmd2_gamma_1.0        0.0000  0.0000
wasserstein_distance  6.7904  0.0625
R2_feature_means      0.7514  0.0738
